In [ ]:
import os
import json
import random
import time
import requests
from datetime import datetime

# TMDB v3 uses the `api_key` query parameter on requests below.
# Set once in your shell: export TMDB_API_KEY="your_key"
API_KEY = 'NOCHANCE'

BASE_URL = "https://api.themoviedb.org/3"
# w342: good balance for VLMs — smaller than original, consistent max width
IMG_BASE = "https://image.tmdb.org/t/p/w342"

# Throttle to stay under TMDB limits (API ~40 req/10s; CDN is separate but be polite)
API_DELAY_SEC = 0.25
IMAGE_DELAY_SEC = 0.15
MAX_429_RETRIES = 5


def get_with_throttle(url, *, params=None, delay=API_DELAY_SEC, timeout=30):
    """GET with spacing and simple retry on HTTP 429."""
    for attempt in range(MAX_429_RETRIES):
        time.sleep(delay)
        r = requests.get(url, params=params, timeout=timeout)
        if r.status_code == 429:
            wait = int(r.headers.get("Retry-After", 2 + attempt * 2))
            time.sleep(wait)
            continue
        r.raise_for_status()
        return r
    raise RuntimeError(f"Too many 429 responses for {url}")

In [2]:
def get_genres():
    url = f"{BASE_URL}/genre/movie/list"
    r = get_with_throttle(url, params={"api_key": API_KEY})
    data = r.json()["genres"]
    return {g["id"]: g["name"] for g in data}

genre_map = get_genres()
print(genre_map)

{28: 'Action', 12: 'Adventure', 16: 'Animation', 35: 'Comedy', 80: 'Crime', 99: 'Documentary', 18: 'Drama', 10751: 'Family', 14: 'Fantasy', 36: 'History', 27: 'Horror', 10402: 'Music', 9648: 'Mystery', 10749: 'Romance', 878: 'Science Fiction', 10770: 'TV Movie', 53: 'Thriller', 10752: 'War', 37: 'Western'}


In [3]:
def get_movies(genre_id, pages=5):
    movies = []
    for p in range(1, pages + 1):
        url = f"{BASE_URL}/discover/movie"
        params = {
            "api_key": API_KEY,
            "with_genres": genre_id,
            "page": p,
            "sort_by": "popularity.desc",
            "include_adult": False
        }
        r = get_with_throttle(url, params=params)
        movies.extend(r.json().get("results", []))
    return movies

In [4]:
def get_decade(date_str):
    try:
        year = int(date_str[:4])
        decade = (year // 10) * 10
        return f"{decade}s"
    except:
        return None

In [5]:
GENRE_QUESTIONS = [
    "What genre does this movie belong to?",
    "Which genre best describes this film?",
    "What is the primary genre of this movie?"
]

DECADE_QUESTIONS = [
    "When was this movie released (decade)?",
    "Which decade is this movie from?",
    "In what decade was this film released?"
]

In [6]:
from PIL import Image
from io import BytesIO


def download_poster(url, path, timeout=30):
    """Download poster bytes as served by TMDB (w342; no local resize)."""
    r = get_with_throttle(url, delay=IMAGE_DELAY_SEC, timeout=timeout)
    content = r.content
    with open(path, "wb") as f:
        f.write(content)
    with Image.open(BytesIO(content)) as im:
        w, h = im.size
    print(f"{path}: {w} x {h}")

In [7]:
def build_dataset(target_per_class=50):
    os.makedirs("image", exist_ok=True)

    genre_map = get_genres()
    reverse_genre = {v: k for k, v in genre_map.items()}

    selected_genres = ["Action", "Comedy", "Horror", "Romance", "Crime", "Documentary"]

    jsonl_lines = []
    img_id = 0

    for genre_name in selected_genres:
        genre_id = reverse_genre[genre_name]
        movies = get_movies(genre_id, pages=10)

        count = 0
        seen = set()

        for m in movies:
            if count >= target_per_class:
                break

            if not m.get("poster_path"):
                continue

            movie_id = m["id"]
            if movie_id in seen:
                continue
            seen.add(movie_id)

            # --- image ---
            img_path = f"image/{img_id:04d}.jpg"
            url = IMG_BASE + m["poster_path"]

            try:
                download_poster(url, img_path)
            except Exception:
                continue

            # --- labels ---
            genre_label = genre_name.lower()

            decade = get_decade(m.get("release_date", ""))

            # --- generate multiple QA pairs ---
            qa_pairs = []

            # genre questions
            for q in GENRE_QUESTIONS:
                qa_pairs.append({
                    "image": img_path,
                    "question": q,
                    "answer": genre_label
                })

            # decade questions
            if decade:
                for q in DECADE_QUESTIONS:
                    qa_pairs.append({
                        "image": img_path,
                        "question": q,
                        "answer": decade
                    })

            jsonl_lines.extend(qa_pairs)

            img_id += 1
            count += 1

    # write JSONL
    with open("data.jsonl", "w") as f:
        for line in jsonl_lines:
            f.write(json.dumps(line) + "\n")

    print(f"Saved {len(jsonl_lines)} QA pairs")

In [8]:
build_dataset()

image/0000.jpg: 342 x 513
image/0001.jpg: 342 x 513
image/0002.jpg: 342 x 513
image/0003.jpg: 342 x 513
image/0004.jpg: 342 x 513
image/0005.jpg: 342 x 513
image/0006.jpg: 342 x 513
image/0007.jpg: 342 x 513
image/0008.jpg: 342 x 513
image/0009.jpg: 342 x 513
image/0010.jpg: 342 x 513
image/0011.jpg: 342 x 513
image/0012.jpg: 342 x 513
image/0013.jpg: 342 x 513
image/0014.jpg: 342 x 513
image/0015.jpg: 342 x 513
image/0016.jpg: 342 x 513
image/0017.jpg: 342 x 513
image/0018.jpg: 342 x 513
image/0019.jpg: 342 x 513
image/0020.jpg: 342 x 513
image/0021.jpg: 342 x 513
image/0022.jpg: 342 x 513
image/0023.jpg: 342 x 513
image/0024.jpg: 342 x 513
image/0025.jpg: 342 x 513
image/0026.jpg: 342 x 482
image/0027.jpg: 342 x 513
image/0028.jpg: 342 x 513
image/0029.jpg: 342 x 513
image/0030.jpg: 342 x 513
image/0031.jpg: 342 x 513
image/0032.jpg: 342 x 513
image/0033.jpg: 342 x 513
image/0034.jpg: 342 x 513
image/0035.jpg: 342 x 513
image/0036.jpg: 342 x 513
image/0037.jpg: 342 x 513
image/0038.j

In [12]:
import pandas as pd


df = pd.read_json("data.jsonl", lines=True)

In [13]:
df['answer'].value_counts()

answer
2020s          591
action         150
comedy         150
horror         150
romance        150
crime          150
documentary    150
2010s          138
2000s           57
1990s           51
1980s           39
1970s           15
1940s            3
1950s            3
Name: count, dtype: int64